# NRP USCMS Analysis Hub

**[website version](https://training.nrp-nautilus.io/cms-hats/6_analysis_hub.html)** — most of this episode is interactive terminal work (password prompts), so this notebook mostly points you to a terminal; only the verification and cleanup steps at the end are runnable cells.

This lesson covers the **grid certificate** setup for the NRP USCMS Analysis Hub ([uscms-af.nrp-nautilus.io](https://uscms-af.nrp-nautilus.io)) — the piece that lets your jobs pull data from CMS's grid storage. `kubectl` access itself is covered on the [setup page](../../lessons/0_setup.md) via `grid-kube-setup`.

## Setting up your grid certificate

You'll need your CERN grid certificate exported as a `.p12` file. Drag it into the JupyterLab file browser on the left (or use the **Upload** button) before continuing.

### 1. Import the certificate

**🖥️ Terminal step** — prompts for your import password and a PEM pass phrase, so it can't run as a notebook cell. Open a terminal (**File → New → Terminal**):

```bash
grid-cert-import
```

1. **The import password** — the one you chose when exporting the `.p12` file.
2. **A PEM pass phrase** — protects the key at rest on the hub. You'll type it every time you create a proxy, so keeping it the same as the import password is fine.

<details>
<summary>Expected output</summary>

```text
jovyan@jupyter-...:~$ grid-cert-import
Importing: /home/jovyan/myCertificate.p12
  existing usercert.pem -> usercert.pem.20260804150152.bak
  existing userkey.pem -> userkey.pem.20260804150152.bak

Enter Import Password:
Enter Import Password:
Enter PEM pass phrase:
Verifying - Enter PEM pass phrase:

Installed:
  subject=DC=ch, DC=cern, OU=Organic Units, OU=Users, CN=ddiaz, CN=821822, CN=Daniel Diaz
  notBefore=Aug  4 05:43:37 2026 GMT
  notAfter=Sep  8 05:43:37 2027 GMT

Next:   grid-proxy-init
```
</details>

### 2. Create your proxy

**🖥️ Terminal step** — prompts for the PEM pass phrase:

```bash
grid-proxy-init
```

Contacts the CMS VOMS server and writes a short-lived proxy to `~/.globus/x509up`. A `voms2-...` server occasionally times out on the first attempt — `grid-proxy-init` retries automatically, so a retry line in the output is expected, not a failure.

<details>
<summary>Expected output</summary>

```text
jovyan@jupyter-...:~$ grid-proxy-init
Enter GRID pass phrase for this identity:
Contacting  voms2-cms-auth.cern.ch:443 [/DC=ch/DC=cern/OU=computers/CN=voms2-cms-auth.cern.ch] "cms"...
Error contacting  voms-cms-auth.cern.ch:443 for VO cms: voms-cms-auth.cern.ch
Contacting  voms2-cms-auth.cern.ch:443 [/DC=ch/DC=cern/OU=computers/CN=cms-auth.cern.ch] "cms"...
Remote VOMS server contacted succesfully.

Created proxy in /home/jovyan/.globus/x509up.

Your proxy is valid until Wed Aug 12 15:02:24 UTC 2026

  /DC=ch/DC=cern/OU=Organic Units/OU=Users/CN=ddiaz/CN=821822/CN=Daniel Diaz/CN=1734779629
  691198
  cms

Proxy written to /home/jovyan/.globus/x509up
To use it from pods in other namespaces:  grid-proxy-publish <namespace> [...]
```
</details>

![Grid certificate import and proxy creation in a hub terminal](../../images/grid-cert.png)

### 3. Verify it

In [ ]:
xrdcp root://cmsxrootd.fnal.gov//store/group/lpclonglived/B-ParkingLLPs/keep.txt .


If the copy succeeds, your proxy is good — this is the same mechanism the [CMS Data Access](5_cms_data.ipynb) notebook's `xrdcp` step relies on, just via a Kubernetes Secret instead of directly from the hub terminal.

## Sharing your proxy with another namespace

`grid-proxy-publish` copies your proxy into a namespace as a Secret, so Jobs there can mount it:

In [ ]:
grid-proxy-publish <namespace>   # ✏️ EDIT to your own personal namespace


**⚠️ Only publish to your own personal namespace**

**Never** run `grid-proxy-publish` against a shared or team namespace — only your own personal one.

Any member of a namespace can read that namespace's Secrets. If you publish your proxy to a shared namespace, every other member can use *your* proxy — acting as you against CMS grid storage — without your knowledge. Since a grid proxy is tied to your personal identity and the CERN Certificate Authority's usage policy holds *you* responsible for whatever it's used for, sharing access this way is a policy violation even if nothing goes wrong technically.

If a Job in a shared namespace needs grid data access, have the person who runs that Job publish their **own** proxy there — don't publish yours on their behalf.

## Clean up

Proxies are short-lived by design, so there's nothing to revoke. If you published a proxy to a namespace you don't want it in anymore, find the Secret it created and delete it:

In [ ]:
kubectl get secrets -n <namespace>   # ✏️ EDIT to the namespace you published to


In [ ]:
kubectl delete secret -n <namespace> <secret-name>   # ✏️ EDIT both placeholders


---

## ✅ Check your work

Verifies the state of your resources on the cluster — rerun any time.

In [ ]:
bash check.sh 6
